# Sustentación en Colab — modelo vigente (`v1.1.0`)

Restauración de fotografías con Pix2Pix · Redes Neuronales y Deep Learning (UAO).

Este notebook **no entrena**: reproduce en vivo las cifras de la sustentación con el modelo publicado
en el release `v1.1.0` y abre la demo. El entrenamiento está en `pix2pix_colab.ipynb` y la corrida
original de 20 épocas, con sus salidas, en `pix2pix_entrenamiento_colab.ipynb`.

| Celda | Qué hace | Tiempo aprox. |
|---|---|---|
| 1 | Clona el repositorio e instala dependencias | 1–2 min |
| 2 | Monta Drive, descomprime `datasetv1.zip` y valida el dataset | 1 min |
| 3 | Descarga el modelo del release `v1.1.0` | 30 s |
| 4 | Evalúa las 45 fotos de prueba (debe dar 21.30 dB con TTA) | 1 min con GPU, ~5 min sin GPU |
| 5 | Curvas de entrenamiento, galería y peores casos | 10 s |
| 6 | Demo con Gradio (enlace público temporal) | 30 s |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (opcional:
sin GPU también funciona, más lento). Luego *Entorno de ejecución → Ejecutar todas*, o celda por celda.

## 1. Repositorio y dependencias

In [ ]:
import os, re, subprocess, sys, zipfile
from pathlib import Path

REPO = Path('/content/Trabajo-Final-de-Redes')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '-q', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Isabela1105/Trabajo-Final-de-Redes.git', str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Commit:', subprocess.check_output(['git', 'log', '-1', '--format=%h %s'], text=True).strip())

# Colab ya trae TensorFlow (con GPU): no se reinstala; el resto sin versiones rígidas.
paquetes = []
for linea in Path('requirements.txt').read_text(encoding='utf-8').splitlines():
    linea = linea.split('#')[0].strip()
    if linea:
        nombre = re.split(r'[<>=!~\[]', linea)[0].strip()
        if not nombre.lower().startswith('tensorflow'):
            paquetes.append(nombre)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *paquetes], check=True)

import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'no (CPU)')

## 2. Dataset (`datasetv1.zip`)

Busca `datasetv1.zip` en tu Google Drive (o un acceso directo a la carpeta del equipo). Si no lo
encuentra, abre un botón para subirlo desde tu computador. Debe terminar en **Dataset OK**.

In [ ]:
from google.colab import drive, files

drive.mount('/content/drive')
zips = sorted(Path('/content').glob('datasetv1*.zip')) or sorted(
    Path('/content/drive/MyDrive').glob('**/datasetv1*.zip'))
if not zips:
    print('No encontré datasetv1.zip en Drive: súbelo aquí.')
    zips = [Path('/content') / nombre for nombre in files.upload()]
with zipfile.ZipFile(zips[0]) as z:
    tops = {n.split('/')[0] for n in z.namelist() if n.strip() and not n.startswith('__MACOSX')}
    z.extractall(REPO if 'data' in tops else REPO / 'data')
print('Dataset:', zips[0])
subprocess.run([sys.executable, 'scripts/check_dataset.py'], check=True)

## 3. Modelo vigente (release `v1.1.0`)

Generador de la época 48, elegido por la menor L1 de validación (`best.json`).

In [ ]:
URL = ('https://github.com/Isabela1105/Trabajo-Final-de-Redes/releases/download/'
       'v1.1.0/pix2pix_100ep_mejor.zip')
CKPT = REPO / 'outputs' / 'pix2pix_100ep_mejor' / 'best'
if not CKPT.is_dir():
    subprocess.run(['wget', '-q', '-O', '/content/modelo.zip', URL], check=True)
    zipfile.ZipFile('/content/modelo.zip').extractall(REPO)
print((CKPT / 'best.json').read_text())

## 4. Métricas en las 45 fotos de prueba

Debe dar exactamente: Entrada 20.30 dB · Mediana 20.05 · Pix2Pix 21.14 · **Pix2Pix + TTA 21.30 dB,
SSIM 0.7165, L1 0.0681**. Después, el análisis de errores por nivel y tipo de daño.

In [ ]:
import pandas as pd

NOMBRE = 'Pix2Pix 100 épocas (mejor val., ép. 48)'
EVAL = REPO / 'outputs' / 'eval_sustentacion'
subprocess.run([sys.executable, '-m', 'src.gan.evaluate', '--config', 'configs/base.yaml',
                '--checkpoint', str(CKPT), '--name', NOMBRE, '--tta', '--out', str(EVAL)], check=True)
display(pd.read_csv(EVAL / 'summary.csv').round(4))
print(subprocess.check_output([sys.executable, 'scripts/error_analysis.py', '--per-image',
                               str(EVAL / 'per_image.csv'), '--model', NOMBRE + ' + TTA'], text=True))

## 5. Curvas de entrenamiento, galería y peores casos

In [ ]:
from IPython.display import Image, display

for titulo, ruta in [('Curvas de entrenamiento (mejor validación en la época 48)', REPO / 'docs/img/curvas_100ep.png'),
                     ('Resultados en prueba', EVAL / 'results_grid.png'),
                     ('Peores casos (elegidos automáticamente)', EVAL / 'failures.png')]:
    print(titulo)
    display(Image(filename=str(ruta), width=900))

## 6. Demo

Abre la demo aquí abajo y genera un enlace público temporal (`gradio.live`). Las fotos de prueba
están en `data/processed/test/input` (por ejemplo `0815_v1.png`, `0898_v2.png`, `0842_v1.png`);
descárgalas desde el panel de archivos (izquierda) o arrastra cualquier foto. Detén la celda (■) al
terminar.

In [ ]:
from app.gradio_app import build_interface

build_interface(str(CKPT)).launch(share=True)